<a href="https://colab.research.google.com/github/Muru-Pan/NLP/blob/main/NLP_prj(Text_rewritting)_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers sentencepiece torch -q

In [ ]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

MODEL_NAME   = "humarin/chatgpt_paraphraser_on_T5_base"
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model on: {TORCH_DEVICE}")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(TORCH_DEVICE)
print("Model loaded successfully!")


In [ ]:
def get_paraphrases(
    input_text: str,
    num_beams: int = 10,
    num_return_sequences: int = 10,
    max_length: int = 128
):
    if num_return_sequences > num_beams:
        raise ValueError("num_return_sequences must be <= num_beams")

    # T5 paraphrase models expect this prefix
    prompt = f"paraphrase: {input_text} </s>"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length,
    ).to(TORCH_DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=num_beams,
            num_return_sequences=num_return_sequences,
            no_repeat_ngram_size=2,   # avoids repetitive output
            early_stopping=True,
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [ ]:
sentence = input("Enter the sentence you want to rewrite: ")

paraphrases = get_paraphrases(sentence, num_beams=10, num_return_sequences=10)

print(f"\n{'─'*50}")
print(f"Original : {sentence}")
print(f"{'─'*50}")
for i, para in enumerate(paraphrases, 1):
    print(f"{i:>2}. {para}")